# VL13 - Parameter-Efficient Fine-Tunning 
# Part 2 - Evaluation
This is the second part of the PEFT fine-tunning lab. This notebook assumes you have already performed the setup of your environment, as well as the fine-tunning in the Part1 notebook.

In [ ]:
import os
import re
import json
import random
import numpy as np
import torch

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

## 1. Reloading and preparing the data 
### 1.1 Load and prepare dataset
We should download and prepare the model with the same pipeline as in the Part1, except for the parts that were specific to the fine-tunning. 

Below we load and normalise the dataset.

In [ ]:
import json

PATH = "../../data/sroie/icdar-2019-sroie.json"

with open(PATH, "r", encoding="utf-8") as f:
    sroie = json.load(f)

FIELDS = ["company", "date", "address", "total"] # fields to be extracted

items = sroie["items"]
items[0].keys()


In [ ]:
def normalize_fields(fields: dict):
    """
    Ensure:
    - all required keys exist
    - missing values are marked as NOT ANSWERABLE
    - output is a JSON string (stable format)
    """
    out = {}
    for f in FIELDS:
        val = fields.get(f)
        if val is None or str(val).strip() == "":
            out[f] = "NOT ANSWERABLE"
        else:
            out[f] = str(val).strip()
    return json.dumps(out, ensure_ascii=False)

def build_receipt_examples(items):
    examples = []
    for it in items:
        examples.append({
            "text": it["transcription"],
            "gold_json": normalize_fields(it["fields"])
        })
    return examples

receipts_gold = build_receipt_examples(items)

print("Num examples:", len(receipts_gold))
print("\nExample:")
print("TEXT:\n", receipts_gold[0]["text"][:400])
print("GOLD JSON:\n", receipts_gold[0]["gold_json"])


### 1.2 Train/test split

We split the labeled receipts into:
- training set (used to learn)
- test set (held out, used for evaluation)

We keep the split fixed using a random seed so results are reproducible. Same seed as in Part1.


In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)

ds = Dataset.from_list(receipts_gold).train_test_split(test_size=0.2, seed=seed)
train_ds, test_ds = ds["train"], ds["test"]

print("Train size:", len(train_ds))
print("Test size:", len(test_ds))


## 2. Preparing for inference

### 2.1 Define the baseline instructions

We reuse the system and instructions that we use in the fine-tunning process. These will be used to build the prompts.


In [ ]:
SYSTEM = "You extract structured information from receipts."

INSTRUCTION = """Extract the following fields from the receipt:
- company
- date
- address
- total

Return ONLY valid JSON with exactly these keys:
company, date, address, total

If a field is missing, use NOT ANSWERABLE for that field.
Do not add extra keys. Do not add explanations.
"""

def build_messages(receipt_text, system, instruction, target_json=None):
    msgs = [
        {"role": "system", "content": system},
        {"role": "user", "content": instruction + "\n\nRECEIPT:\n" + receipt_text},
    ]
    if target_json is not None:
        msgs.append({"role": "assistant", "content": target_json})
    return msgs


### 2.2 Inference function 

We define a function `generate_json` that:
- builds the prompt based on input instructions
- runs `model.generate()`
- returns the raw generated text

We keep generation settings conservative to reduce hallucination.

In [ ]:
def generate_json(receipt_text, model, tokenizer, system, instruction, max_new_tokens=200):
    msgs = build_messages(receipt_text, system, instruction)
    try:
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except Exception:
        prompt = f"[SYSTEM]\n{SYSTEM}\n[USER]\n{INSTRUCTION}\n\nRECEIPT:\n{receipt_text}\n[ASSISTANT]\n"

    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            top_p=1.0,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    gen = out[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(gen, skip_special_tokens=True)

    # Return only the part after the prompt if possible
    return text.strip()


### 2.3 Parse model output and evaluate

LLMs sometimes output extra text.
We try to extract the first JSON object from the output.
Then we can compare predicted vs gold fields.

This mirrors last week's evaluation: we want correct fields AND valid JSON.

In [ ]:
# Extract json from model output
def extract_json_object(text: str):
    # Find the first {...} block (simple but works well for labs)
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return None
    candidate = m.group(0)
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        return None

# Normalise values for comparison
def normalize_value(v):
    if v is None:
        return None
    v = str(v).strip()
    v = re.sub(r"\s+", " ", v)
    return v

FIELDS = ["company", "date", "address", "total"]

# Comput exact matches
def field_em(pred_obj, gold_obj):
    # exact match per field (simple baseline)
    em = {}
    for f in FIELDS:
        pv = normalize_value(pred_obj.get(f, None)) if pred_obj else None
        gv = normalize_value(gold_obj.get(f, None)) if gold_obj else None
        em[f] = int(pv == gv)
    return em

## 3. Loading model
The helper below allow us to load a model and it's LoRA 'adapter' if available. We will use this function to load the model and the fine-tuned weights. 

### 3.1 Loading model and tokenizer

In [ ]:
def load_model_and_tokenizer(base_model_path, adapter_path=None, use_4bit=True):
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel, prepare_model_for_kbit_training    
    tokenizer = AutoTokenizer.from_pretrained(base_model_path, use_fast=True)
    tokenizer.pad_token = tokenizer.eos_token

    bnb_config = None
    if use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )

    model = AutoModelForCausalLM.from_pretrained(
        base_model_path,
        device_map="auto",
        quantization_config=bnb_config,
        dtype=torch.bfloat16,
    )

    if use_4bit:
        model = prepare_model_for_kbit_training(model)

    if adapter_path:
        model = PeftModel.from_pretrained(model, adapter_path)

    model.eval()
    return model, tokenizer

### 2.4 Testing inference
Let's load the model, and fine-tuned wieghts to run an inference test.

In [ ]:
model, tokenizer = load_model_and_tokenizer("../../models/Llama-3.2-1B-Instruct", 
                                            "receipt_lora_adapter", 
                                            use_4bit=True)

In [ ]:
# quick smoke test on 1 example
i = 0
raw = generate_json(test_ds[i]["text"], model, tokenizer, SYSTEM, INSTRUCTION)
pred = extract_json_object(raw)
gold = json.loads(test_ds[i]["gold_json"])

print("RAW OUTPUT:\n", raw)
print("\nPARSED JSON:\n", pred)
print("\nGOLD JSON:\n", gold)
print("\nFIELD EM:\n", field_em(pred, gold))

### 3. Evaluate fine-tuned model on the test set

We run the fine-tuned model on the held-out receipts and compute:
- JSON validity rate (did we parse JSON?)
- exact match per field
- all-fields exact match (all four correct)

This gives a clean before/after comparison with last week's prompting baseline.


In [ ]:
from tqdm.auto import tqdm
import time
import json

def evaluate_model(
    model, tokenizer, dataset, system, instruction,
    max_items=None, show_progress=True,
    return_predictions=False,
    store_raw=True,
):
    n = len(dataset) if max_items is None else min(len(dataset), max_items)
    valid_json = 0
    per_field = {f: 0 for f in FIELDS}
    all_fields = 0

    predictions = []  # will stay empty unless return_predictions=True

    it = range(n)
    if show_progress:
        it = tqdm(it, total=n, desc="Evaluating", unit="ex")

    t0 = time.time()

    for i in it:
        ex = dataset[i]
        raw = generate_json(ex["text"], model, tokenizer, system, instruction)
        pred = extract_json_object(raw)
        gold = json.loads(ex["gold_json"])

        if pred is not None:
            valid_json += 1

        em = field_em(pred, gold)
        for f in FIELDS:
            per_field[f] += em[f]

        all_correct = (sum(em.values()) == len(FIELDS))
        if all_correct:
            all_fields += 1

        # Collect per-example record for inspection
        if return_predictions:
            record = {
                "i": i,
                "text": ex["text"],
                "gold": gold,
                "pred": pred,
                "em": em,
            }
            if store_raw:
                record["raw"] = raw

            # Optional quick error label
            if pred is None:
                record["error_type"] = "no_json"
            else:
                missing = [f for f in FIELDS if f not in pred]
                record["error_type"] = "missing_keys" if missing else "ok_or_value_error"
                record["missing_keys"] = missing

            predictions.append(record)

        # Progress display
        if show_progress and (i + 1) % 5 == 0:
            it.set_postfix({
                "json_valid": f"{valid_json/(i+1):.2f}",
                "all_em": f"{all_fields/(i+1):.2f}",
                "sec/ex": f"{(time.time()-t0)/(i+1):.2f}",
            })

    results = {
        "n": n,
        "json_valid_rate": valid_json / n,
        "all_fields_em": all_fields / n,
    }
    for f in FIELDS:
        results[f"EM_{f}"] = per_field[f] / n

    if return_predictions:
        return results, predictions
    return results


In [ ]:
results_ft = evaluate_model(model, tokenizer, test_ds, SYSTEM, INSTRUCTION, max_items=50)
print(results_ft)

## 4. Compare models
We can compare the base model with the LoRA version to assess the performance. 

Answer these questions using your results:

1) Did JSON validity improve compared to prompting?
2) Which fields improved most, and which did not?
3) What kinds of errors remain?
4) Was fine-tuning worth it compared to prompt engineering?


In [ ]:
models_to_compare = [
    {"name": "base-1b", "base": "../../models/Llama-3.2-1B-Instruct", "adapter": None},
    #{"name": "lora-1b", "base": "../../models/Llama-3.2-1B-Instruct", "adapter": "receipt_lora_adapter"},
]

results = {}
for m in models_to_compare:
    model, tokenizer = load_model_and_tokenizer(m["base"], m["adapter"], use_4bit=True)
    result = evaluate_model(model, tokenizer, test_ds, SYSTEM, INSTRUCTION, max_items=50, return_predictions=True)
    results[m["name"]] = result
    metrics = result[0] if isinstance(result, tuple) else result
    print(m["name"], metrics)


### 4.1 Impact of prompting on the base model
We can try changing the instructions to see if we can improve the base model by changing the prompting strategy. Below we have differnt n-shot alternatives. You can try more.

In [ ]:
import json

BASE_INSTRUCTION = INSTRUCTION

FEWSHOT_EXAMPLES = [
    # Example A: DATE with hyphens + explicit "TOTAL:" + multi-line address
    {
        "receipt": """ONE ONE THREE SEAFOOD RESTAURANT SDN BHD
(1120908-M)
NO.1, TAMAN SRI DENGKIL, JALAN AIR HITAM
43800 DENGKIL, SELANGOR.
TAX INVOICE
DATE : 20-06-2018 21:36:11
TOTAL (INCLUSIVE OF GST): 38.00
TOTAL: 38.00""",
        "json": {
            "company": "ONE ONE THREE SEAFOOD RESTAURANT SDN BHD",
            "date": "20-06-2018",
            "address": "NO.1, TAMAN SRI DENGKIL, JALAN AIR HITAM 43800 DENGKIL, SELANGOR.",
            "total": "38.00",
        },
    },

    # Example B: DATE with slashes + invoice formatting + multiple totals (pick INCL)
    {
        "receipt": """GL HANDICRAFT & TAIL ORING
19, JALAN KANCIL,
OFF JALAN PUDU,
55100 KUALA LUMPUR
MALAYSIA
TAX INVOICE
INVOICE NO.: CS 10012
DATE : 20/03/2018 13:01
SUBTOTAL : 102.00
TOTAL EXCL. OF GST 96.23
TOTAL INCL. OF GST 102.00
TOTAL AMT ROUNDED 102.00""",
        "json": {
            "company": "GL HANDICRAFT & TAIL ORING",
            "date": "20/03/2018",
            "address": "19, JALAN KANCIL, OFF JALAN PUDU, 55100 KUALA LUMPUR MALAYSIA",
            "total": "102.00",
        },
    },

    # Example C: Small receipt + ambiguous lines + reinforce NOT ANSWERABLE option
    # (Here we simulate missing address details beyond city line; you can replace with a real missing-field case if present.)
    {
        "receipt": """LIAN HING STATIONERY SDN BHD
(162761-M)
TAX INVOICE
27/03/2018 NO : CS-20243
TOTAL AMT PAYABLE : 12.00""",
        "json": {
            "company": "LIAN HING STATIONERY SDN BHD",
            "date": "27/03/2018",
            "address": "NOT ANSWERABLE",
            "total": "12.00",
        },
    },
]

def build_instruction_nshot(n_shot: int):
    assert n_shot in (0, 1, 3), "Use n_shot = 0, 1, or 3 for this lab."
    if n_shot == 0:
        return BASE_INSTRUCTION

    demos = FEWSHOT_EXAMPLES[:n_shot]
    demo_blocks = []
    for ex in demos:
        demo_blocks.append(
            "EXAMPLE\n"
            "RECEIPT:\n"
            f"{ex['receipt']}\n"
            "OUTPUT:\n"
            f"{json.dumps(ex['json'], ensure_ascii=False)}\n"
        )

    return (
        BASE_INSTRUCTION
        + "\n\nHere are examples:\n\n"
        + "\n".join(demo_blocks)
        + "\nNow do the same for the next receipt."
    )


In [ ]:
prompts = [
    #{"name": "0-shot_default", "system": SYSTEM, "instruction": build_instruction_nshot(0)},
    {"name": "1-shot",         "system": SYSTEM, "instruction": build_instruction_nshot(1)},
    {"name": "3-shot",         "system": SYSTEM, "instruction": build_instruction_nshot(3)},
]

m = models_to_compare[0]
model, tokenizer = load_model_and_tokenizer(m["base"], m["adapter"], use_4bit=True)

results_p = {}
for p in prompts:
    result = evaluate_model(model, tokenizer, test_ds, p["system"], p["instruction"], max_items=50, return_predictions=True)
    results_p[p["name"]] = result
    metrics = result[0] if isinstance(result, tuple) else result
    print(p["name"], metrics)
      

### 4.2 Inspect errors
The helper below can help you inspect the output of the models, and possibly refine your prompting strategy.

In [ ]:
def inspect_errors(
    predictions,
    max_examples=5,
    show_raw_chars=500,
    only_error_types=None,
    require_field_errors=False,
):
    """
    Inspect prediction errors in a readable way.

    Parameters
    ----------
    predictions : list of dict
        Output of evaluate_model(..., return_predictions=True)
    max_examples : int
        Maximum number of examples to print
    show_raw_chars : int
        Number of characters of raw model output to show
    only_error_types : list or None
        If provided, only show these error_type values (e.g. ["no_json"])
    require_field_errors : bool
        If True, only show cases where at least one field EM == 0
    """

    # Select candidates
    candidates = []
    for p in predictions:
        if only_error_types is not None:
            if p.get("error_type") not in only_error_types:
                continue

        if require_field_errors:
            if sum(p["em"].values()) == len(p["em"]):
                continue

        candidates.append(p)

    print(f"Showing {min(max_examples, len(candidates))} / {len(candidates)} selected cases")

    for p in candidates[:max_examples]:
        print("\n" + "=" * 80)
        print(f"Example {p['i']}")
        print("Error type:", p.get("error_type"))
        print("Field EM:", p["em"])

        print("\nGOLD:")
        print(p["gold"])

        print("\nPRED:")
        print(p["pred"])

        if "raw" in p:
            print("\nRAW MODEL OUTPUT (truncated):")
            print(p["raw"][:show_raw_chars])

        if "missing_keys" in p and p["missing_keys"]:
            print("\nMissing keys:", p["missing_keys"])


In [ ]:
inspect_errors(
    results["3-shot"][1],
    max_examples=5,
    show_raw_chars=100000
)